# PeMS04 Exploratory Data Analysis

**Project:** Spatio-Temporal Machine Learning for Urban Traffic Congestion Prediction and Dynamic Risk Mapping

**Dataset:** PeMS04 — Caltrans Performance Measurement System, District 4 (San Francisco Bay Area)

**Period:** January – February 2018 (59 days)

**Resolution:** 5-minute intervals (12 samples/hour)

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
from datetime import datetime, timedelta

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
FIG_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Figures dir:', FIG_DIR)

Project root: C:\Users\Lenovo\OneDrive\Desktop\AUCML\traffic-congestion-ml
Figures dir: C:\Users\Lenovo\OneDrive\Desktop\AUCML\traffic-congestion-ml\outputs\figures


## 1. Load Dataset

In [2]:
# Load PeMS04 data
npz_path = os.path.join(RAW_DIR, 'pems04.npz')
raw_data = np.load(npz_path)

print('Keys in npz file:', list(raw_data.keys()))
data = raw_data['data']
print(f'\nData shape: {data.shape}')
print(f'  Time steps: {data.shape[0]}')
print(f'  Sensors: {data.shape[1]}')
print(f'  Features: {data.shape[2]}')
print(f'  Dtype: {data.dtype}')
print(f'\nFeature channels:')
print(f'  Channel 0: Traffic Flow (vehicles per 5 minutes)')
print(f'  Channel 1: Occupancy (proportion 0-1)')
print(f'  Channel 2: Speed (miles per hour)')

# Create time index (Jan 1 - Feb 28, 2018)
start_time = datetime(2018, 1, 1, 0, 0)
time_index = [start_time + timedelta(minutes=5*i) for i in range(data.shape[0])]
print(f'\nTime range: {time_index[0]} to {time_index[-1]}')
print(f'Duration: {(time_index[-1] - time_index[0]).days} days')

Keys in npz file: ['data']



Data shape: (16992, 307, 3)
  Time steps: 16992
  Sensors: 307
  Features: 3
  Dtype: float64

Feature channels:
  Channel 0: Traffic Flow (vehicles per 5 minutes)
  Channel 1: Occupancy (proportion 0-1)
  Channel 2: Speed (miles per hour)

Time range: 2018-01-01 00:00:00 to 2018-02-28 23:55:00
Duration: 58 days


## 2. Descriptive Statistics

In [3]:
feature_names = ['Flow (veh/5min)', 'Occupancy (ratio)', 'Speed (mph)']

for i, name in enumerate(feature_names):
    feat = data[:, :, i]
    print(f'\n{"="*50}')
    print(f'{name}')
    print(f'{"="*50}')
    print(f'  Shape: {feat.shape}')
    print(f'  Min:    {feat.min():.4f}')
    print(f'  Max:    {feat.max():.4f}')
    print(f'  Mean:   {feat.mean():.4f}')
    print(f'  Median: {np.median(feat):.4f}')
    print(f'  Std:    {feat.std():.4f}')
    print(f'  Q25:    {np.percentile(feat, 25):.4f}')
    print(f'  Q75:    {np.percentile(feat, 75):.4f}')
    print(f'  Zeros:  {(feat == 0).sum()} ({(feat == 0).sum()/feat.size*100:.2f}%)')
    print(f'  NaN:    {np.isnan(feat).sum()}')


Flow (veh/5min)
  Shape: (16992, 307)
  Min:    0.0000
  Max:    919.0000
  Mean:   211.7008
  Median: 180.0000
  Std:    158.0684
  Q25:    81.0000


  Q75:    319.0000
  Zeros:  82935 (1.59%)
  NaN:    0

Occupancy (ratio)
  Shape: (16992, 307)
  Min:    0.0000
  Max:    0.7716
  Mean:   0.0528
  Median: 0.0443
  Std:    0.0495


  Q25:    0.0213
  Q75:    0.0691
  Zeros:  83044 (1.59%)
  NaN:    0

Speed (mph)
  Shape: (16992, 307)
  Min:    3.0000
  Max:    85.2000
  Mean:   63.4706
  Median: 65.6000


  Std:    8.3557
  Q25:    62.5000
  Q75:    67.8000
  Zeros:  0 (0.00%)
  NaN:    0


## 3. Missing Value Analysis

In [4]:
print('Missing Value Analysis')
print('=' * 50)

for i, name in enumerate(feature_names):
    feat = data[:, :, i]
    nan_count = np.isnan(feat).sum()
    zero_count = (feat == 0).sum()
    total = feat.size
    print(f'\n{name}:')
    print(f'  NaN values: {nan_count} ({nan_count/total*100:.4f}%)')
    print(f'  Zero values: {zero_count} ({zero_count/total*100:.2f}%)')
    print(f'  Valid non-zero: {total - nan_count - zero_count} ({(total - nan_count - zero_count)/total*100:.2f}%)')

# Zero values per sensor for flow
flow = data[:, :, 0]
zeros_per_sensor = (flow == 0).sum(axis=0)
print(f'\nFlow: Zero observations per sensor:')
print(f'  Min zeros: {zeros_per_sensor.min()} (sensor {zeros_per_sensor.argmin()})')
print(f'  Max zeros: {zeros_per_sensor.max()} (sensor {zeros_per_sensor.argmax()})')
print(f'  Mean zeros: {zeros_per_sensor.mean():.1f}')
print(f'  Sensors with >5% zeros: {(zeros_per_sensor > 0.05 * data.shape[0]).sum()}')

Missing Value Analysis

Flow (veh/5min):
  NaN values: 0 (0.0000%)
  Zero values: 82935 (1.59%)
  Valid non-zero: 5133609 (98.41%)

Occupancy (ratio):
  NaN values: 0 (0.0000%)
  Zero values: 83044 (1.59%)
  Valid non-zero: 5133500 (98.41%)

Speed (mph):
  NaN values: 0 (0.0000%)
  Zero values: 0 (0.00%)
  Valid non-zero: 5216544 (100.00%)

Flow: Zero observations per sensor:
  Min zeros: 0 (sensor 5)
  Max zeros: 6471 (sensor 128)
  Mean zeros: 270.1
  Sensors with >5% zeros: 24


## 4. Traffic Flow Over Time

In [5]:
# Network-average flow over time
flow = data[:, :, 0]
avg_flow = flow.mean(axis=1)

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Full time series
axes[0].plot(time_index, avg_flow, linewidth=0.5, alpha=0.8)
axes[0].set_title('Network-Average Traffic Flow (All 59 Days)', fontsize=14)
axes[0].set_ylabel('Flow (veh/5min)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[0].xaxis.set_major_locator(mdates.WeekdayLocator())

# One week zoom
week_end = 12 * 24 * 7  # 7 days
axes[1].plot(time_index[:week_end], avg_flow[:week_end], linewidth=0.8)
axes[1].set_title('First Week (Jan 1-7, 2018)', fontsize=14)
axes[1].set_ylabel('Flow (veh/5min)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%a %b %d'))

# One day zoom
day_samples = 12 * 24  # 1 day
# Pick a Tuesday (Jan 9 = day 8, index 8*288)
day_start = 8 * day_samples
day_end = day_start + day_samples
axes[2].plot(time_index[day_start:day_end], avg_flow[day_start:day_end], linewidth=1.2)
axes[2].set_title('Single Day — Tuesday, January 9, 2018', fontsize=14)
axes[2].set_ylabel('Flow (veh/5min)')
axes[2].set_xlabel('Time of Day')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'flow_time_series.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: flow_time_series.png')

Saved: flow_time_series.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\2068844128.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Representative Sensor-Level Plots

In [6]:
# Select diverse sensors based on average flow
sensor_means = flow.mean(axis=0)
sorted_sensors = np.argsort(sensor_means)

# Pick low, medium, and high flow sensors
selected_sensors = [
    sorted_sensors[10],                         # Low flow
    sorted_sensors[len(sorted_sensors)//2],     # Medium flow
    sorted_sensors[-10],                        # High flow
]

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Show one week for each sensor
week_end = 12 * 24 * 7
labels = ['Low Flow', 'Medium Flow', 'High Flow']
colors = ['#2196F3', '#4CAF50', '#F44336']

for idx, (sensor, label, color) in enumerate(zip(selected_sensors, labels, colors)):
    axes[idx].plot(time_index[:week_end], flow[:week_end, sensor], 
                   linewidth=0.8, color=color, alpha=0.8)
    axes[idx].set_title(f'Sensor {sensor} — {label} (mean={sensor_means[sensor]:.1f} veh/5min)', fontsize=12)
    axes[idx].set_ylabel('Flow')

axes[-1].set_xlabel('Time')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%a %b %d'))

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'sensor_level_flow.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sensor_level_flow.png')

Saved: sensor_level_flow.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\2886966445.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Feature Distributions

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, color) in enumerate(zip(feature_names, ['#1976D2', '#388E3C', '#D32F2F'])):
    feat = data[:, :, i].flatten()
    # Sample for efficiency
    if len(feat) > 500000:
        sample = np.random.RandomState(42).choice(feat, 500000, replace=False)
    else:
        sample = feat
    
    axes[i].hist(sample, bins=80, color=color, alpha=0.7, edgecolor='white', linewidth=0.5)
    axes[i].set_title(name, fontsize=13)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(np.mean(feat), color='black', linestyle='--', linewidth=1.5, label=f'Mean={np.mean(feat):.2f}')
    axes[i].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

Saved: feature_distributions.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\1450858277.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Sensor-Level Distribution Analysis

In [8]:
# Per-sensor statistics for flow
sensor_stats = pd.DataFrame({
    'sensor_id': range(307),
    'mean_flow': flow.mean(axis=0),
    'std_flow': flow.std(axis=0),
    'max_flow': flow.max(axis=0),
    'min_flow': flow.min(axis=0),
    'zero_pct': (flow == 0).sum(axis=0) / flow.shape[0] * 100,
    'cv': flow.std(axis=0) / (flow.mean(axis=0) + 1e-8),  # coefficient of variation
})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(sensor_stats['sensor_id'], sensor_stats['mean_flow'], width=1.0, color='steelblue', alpha=0.7)
axes[0, 0].set_title('Mean Flow by Sensor')
axes[0, 0].set_xlabel('Sensor ID')
axes[0, 0].set_ylabel('Mean Flow (veh/5min)')

axes[0, 1].bar(sensor_stats['sensor_id'], sensor_stats['std_flow'], width=1.0, color='coral', alpha=0.7)
axes[0, 1].set_title('Flow Standard Deviation by Sensor')
axes[0, 1].set_xlabel('Sensor ID')
axes[0, 1].set_ylabel('Std Flow')

axes[1, 0].scatter(sensor_stats['mean_flow'], sensor_stats['std_flow'], alpha=0.5, s=15)
axes[1, 0].set_title('Mean vs Std Flow (per sensor)')
axes[1, 0].set_xlabel('Mean Flow')
axes[1, 0].set_ylabel('Std Flow')

axes[1, 1].bar(sensor_stats['sensor_id'], sensor_stats['zero_pct'], width=1.0, color='orange', alpha=0.7)
axes[1, 1].set_title('Percentage of Zero-Flow Observations')
axes[1, 1].set_xlabel('Sensor ID')
axes[1, 1].set_ylabel('Zero %')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'sensor_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 highest-flow sensors:')
print(sensor_stats.nlargest(5, 'mean_flow')[['sensor_id', 'mean_flow', 'max_flow', 'std_flow']])
print('\nTop 5 most variable sensors (CV):')
print(sensor_stats.nlargest(5, 'cv')[['sensor_id', 'mean_flow', 'std_flow', 'cv']])
print('\nSaved: sensor_distributions.png')


Top 5 highest-flow sensors:
     sensor_id   mean_flow  max_flow    std_flow
97          97  469.702919     896.0  221.631288
41          41  467.895304     865.0  186.579119
108        108  448.214689     870.0  251.069773
101        101  444.760770     882.0  201.256963
111        111  444.696504     853.0  209.508008

Top 5 most variable sensors (CV):
     sensor_id   mean_flow    std_flow        cv
31          31  119.304614  121.178636  1.015708
226        226   61.626942   62.024402  1.006449
7            7  110.972281  108.508951  0.977802
128        128   47.490290   46.242602  0.973728
266        266   35.712335   31.341114  0.877599

Saved: sensor_distributions.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\782715631.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Temporal Patterns

In [9]:
# Hour-of-day pattern
hours = np.array([t.hour + t.minute/60 for t in time_index])
day_of_week = np.array([t.weekday() for t in time_index])

# Average flow by hour of day
hourly_avg = np.zeros(288)  # 288 = 24*12 five-minute intervals
for i in range(288):
    indices = np.arange(i, len(time_index), 288)
    indices = indices[indices < len(avg_flow)]
    hourly_avg[i] = avg_flow[indices].mean()

hour_ticks = np.arange(288) * 5 / 60  # Convert to hours

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Hour of day
axes[0].plot(hour_ticks, hourly_avg, linewidth=2, color='steelblue')
axes[0].set_title('Average Flow by Time of Day (All Days)', fontsize=13)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Flow (veh/5min)')
axes[0].set_xticks(range(0, 25, 3))
axes[0].axvspan(7, 9, alpha=0.15, color='red', label='AM Peak')
axes[0].axvspan(16, 19, alpha=0.15, color='orange', label='PM Peak')
axes[0].legend()

# Day of week pattern
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_avg = [avg_flow[day_of_week == d].mean() for d in range(7)]
colors = ['steelblue']*5 + ['coral']*2  # Weekday vs weekend

axes[1].bar(dow_names, dow_avg, color=colors, alpha=0.8, edgecolor='white')
axes[1].set_title('Average Flow by Day of Week', fontsize=13)
axes[1].set_ylabel('Avg Flow (veh/5min)')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'temporal_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: temporal_patterns.png')

Saved: temporal_patterns.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\2375916334.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Weekday vs Weekend hourly profiles
weekday_mask = day_of_week < 5
weekend_mask = day_of_week >= 5

# Reshape into (num_days, 288) blocks
num_complete_days = len(avg_flow) // 288
daily_flow = avg_flow[:num_complete_days * 288].reshape(num_complete_days, 288)
daily_dow = day_of_week[::288][:num_complete_days]

weekday_profile = daily_flow[daily_dow < 5].mean(axis=0)
weekend_profile = daily_flow[daily_dow >= 5].mean(axis=0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hour_ticks, weekday_profile, label='Weekday', linewidth=2, color='steelblue')
ax.plot(hour_ticks, weekend_profile, label='Weekend', linewidth=2, color='coral')
ax.set_title('Weekday vs Weekend Traffic Flow Profiles', fontsize=14)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Flow (veh/5min)')
ax.set_xticks(range(0, 25, 2))
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'weekday_weekend_profile.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: weekday_weekend_profile.png')

Saved: weekday_weekend_profile.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\3834473900.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Sensor Correlations

In [11]:
# Correlation matrix for flow (subsample sensors for visibility)
np.random.seed(42)
sample_sensors = sorted(np.random.choice(307, 30, replace=False))

flow_subset = flow[:, sample_sensors]
corr_matrix = np.corrcoef(flow_subset.T)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Flow Correlation Matrix (30 Sample Sensors)', fontsize=14)
ax.set_xlabel('Sensor Index')
ax.set_ylabel('Sensor Index')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'sensor_correlations.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sensor_correlations.png')

Saved: sensor_correlations.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\4042764250.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Adjacency Matrix Analysis

In [12]:
# Load distance/adjacency data
dist_df = pd.read_csv(os.path.join(RAW_DIR, 'distance.csv'))
print('Distance CSV:')
print(f'  Shape: {dist_df.shape}')
print(f'  Columns: {list(dist_df.columns)}')
print(f'\n  Number of edges: {len(dist_df)}')
print(f'  Unique source sensors: {dist_df.iloc[:, 0].nunique()}')
print(f'  Unique dest sensors: {dist_df.iloc[:, 1].nunique()}')
print(f'  Distance range: [{dist_df.iloc[:, 2].min():.1f}, {dist_df.iloc[:, 2].max():.1f}]')
print(f'  Mean distance: {dist_df.iloc[:, 2].mean():.1f}')
print(f'\nSample edges:')
print(dist_df.head(10))

Distance CSV:
  Shape: (340, 3)
  Columns: ['from', 'to', 'cost']

  Number of edges: 340
  Unique source sensors: 281
  Unique dest sensors: 282
  Distance range: [3.2, 2712.1]
  Mean distance: 410.3

Sample edges:
   from   to   cost
0    73    5  352.6
1     5  154  347.2
2   154  263  392.9
3   263   56  440.8
4    56   96  374.6
5    96   42  378.1
6    42   58  364.6
7    58   95  476.8
8    95   72  480.1
9    72  271  419.5


In [13]:
# Build adjacency matrix using Gaussian kernel
all_sensors = sorted(set(dist_df.iloc[:, 0].tolist() + dist_df.iloc[:, 1].tolist()))
sensor_to_idx = {s: i for i, s in enumerate(range(307))}

distances = dist_df.iloc[:, 2].values
sigma = distances.std()
print(f'Gaussian kernel sigma: {sigma:.2f}')

adj_matrix = np.zeros((307, 307))
for _, row in dist_df.iterrows():
    i, j, d = int(row.iloc[0]), int(row.iloc[1]), row.iloc[2]
    if i < 307 and j < 307:
        weight = np.exp(-d**2 / sigma**2)
        adj_matrix[i, j] = weight
        adj_matrix[j, i] = weight  # Symmetric

# Add self-loops
np.fill_diagonal(adj_matrix, 1.0)

print(f'\nAdjacency matrix shape: {adj_matrix.shape}')
print(f'Non-zero entries: {(adj_matrix > 0).sum()}')
print(f'Sparsity: {1 - (adj_matrix > 0).sum() / adj_matrix.size:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Adjacency heatmap
im1 = axes[0].imshow(adj_matrix > 0, cmap='Blues', aspect='equal')
axes[0].set_title('Adjacency Matrix (Binary)', fontsize=13)
axes[0].set_xlabel('Sensor ID')
axes[0].set_ylabel('Sensor ID')

# Distance histogram
axes[1].hist(distances, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_title('Distribution of Road Distances', fontsize=13)
axes[1].set_xlabel('Distance')
axes[1].set_ylabel('Frequency')
axes[1].axvline(distances.mean(), color='red', linestyle='--', label=f'Mean={distances.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'adjacency_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: adjacency_analysis.png')

Gaussian kernel sigma: 257.14

Adjacency matrix shape: (307, 307)
Non-zero entries: 987
Sparsity: 0.9895


Saved: adjacency_analysis.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\1419012634.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Anomaly Detection

In [14]:
# Identify anomalous observations
flow_flat = flow.flatten()
q1 = np.percentile(flow_flat, 25)
q3 = np.percentile(flow_flat, 75)
iqr = q3 - q1
upper_bound = q3 + 3 * iqr

extreme_high = (flow > upper_bound).sum()
print(f'IQR-based outlier detection (flow):')
print(f'  Q1: {q1:.1f}, Q3: {q3:.1f}, IQR: {iqr:.1f}')
print(f'  Upper bound (Q3 + 3*IQR): {upper_bound:.1f}')
print(f'  Values above upper bound: {extreme_high} ({extreme_high/flow.size*100:.3f}%)')

# Check for sudden jumps (consecutive difference > threshold)
flow_diff = np.abs(np.diff(flow, axis=0))
jump_threshold = 5 * flow.std(axis=0)  # per-sensor threshold
jumps = (flow_diff > jump_threshold[np.newaxis, :]).sum()
print(f'\nSudden jumps (>5σ per sensor): {jumps}')

# Sensors with constant flow (possibly faulty)
sensor_ranges = flow.max(axis=0) - flow.min(axis=0)
low_variance = (sensor_ranges < 5).sum()
print(f'Sensors with range < 5: {low_variance}')

# Speed-flow relationship check
speed = data[:, :, 2]
fig, ax = plt.subplots(figsize=(10, 6))
sample_idx = np.random.RandomState(42).choice(flow.size, 50000, replace=False)
ax.scatter(flow.flatten()[sample_idx], speed.flatten()[sample_idx], alpha=0.1, s=1, c='steelblue')
ax.set_xlabel('Flow (veh/5min)', fontsize=12)
ax.set_ylabel('Speed (mph)', fontsize=12)
ax.set_title('Flow-Speed Fundamental Diagram (50k sample)', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'flow_speed_diagram.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: flow_speed_diagram.png')

IQR-based outlier detection (flow):
  Q1: 81.0, Q3: 319.0, IQR: 238.0
  Upper bound (Q3 + 3*IQR): 1033.0
  Values above upper bound: 0 (0.000%)



Sudden jumps (>5σ per sensor): 11
Sensors with range < 5: 0


Saved: flow_speed_diagram.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\3920707379.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Data Split Visualization

In [15]:
# Show chronological split
n = len(avg_flow)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(time_index[:train_end], avg_flow[:train_end], label='Train (60%)', color='steelblue', linewidth=0.5)
ax.plot(time_index[train_end:val_end], avg_flow[train_end:val_end], label='Validation (20%)', color='orange', linewidth=0.5)
ax.plot(time_index[val_end:], avg_flow[val_end:], label='Test (20%)', color='red', linewidth=0.5)

ax.axvline(time_index[train_end], color='black', linestyle='--', linewidth=1.5)
ax.axvline(time_index[val_end], color='black', linestyle='--', linewidth=1.5)

ax.set_title('Chronological Train/Validation/Test Split', fontsize=14)
ax.set_ylabel('Avg Flow (veh/5min)')
ax.legend(loc='upper right', fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'data_split.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSplit boundaries:')
print(f'  Train: {time_index[0]} to {time_index[train_end-1]} ({train_end} steps)')
print(f'  Val:   {time_index[train_end]} to {time_index[val_end-1]} ({val_end - train_end} steps)')
print(f'  Test:  {time_index[val_end]} to {time_index[-1]} ({n - val_end} steps)')
print('\nSaved: data_split.png')


Split boundaries:
  Train: 2018-01-01 00:00:00 to 2018-02-05 09:30:00 (10195 steps)
  Val:   2018-02-05 09:35:00 to 2018-02-17 04:40:00 (3398 steps)
  Test:  2018-02-17 04:45:00 to 2018-02-28 23:55:00 (3399 steps)

Saved: data_split.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25552\3559830.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Summary

In [16]:
print('=' * 60)
print('PeMS04 EDA Summary')
print('=' * 60)
print(f'Dataset: PeMS04 (Caltrans District 4, SF Bay Area)')
print(f'Period: January 1 - February 28, 2018 (59 days)')
print(f'Shape: {data.shape} (timesteps, sensors, features)')
print(f'Sampling: 5-minute intervals (12/hour, 288/day)')
print(f'\nFeatures:')
print(f'  Flow: [{flow.min():.0f}, {flow.max():.0f}] veh/5min, mean={flow.mean():.1f}')
print(f'  Occupancy: [{data[:,:,1].min():.4f}, {data[:,:,1].max():.4f}], mean={data[:,:,1].mean():.4f}')
print(f'  Speed: [{data[:,:,2].min():.1f}, {data[:,:,2].max():.1f}] mph, mean={data[:,:,2].mean():.1f}')
print(f'\nData Quality:')
print(f'  NaN values: {np.isnan(data).sum()}')
print(f'  Zero-flow observations: {(flow == 0).sum()} ({(flow == 0).sum()/flow.size*100:.2f}%)')
print(f'\nAdjacency:')
print(f'  Edges: {len(dist_df)} directed connections')
print(f'  Distance range: [{distances.min():.1f}, {distances.max():.1f}]')
print(f'\nGeographic Coordinates: NOT AVAILABLE in standard release')
print(f'\nForecasting Target: Traffic Flow (channel 0)')
print(f'\nFigures saved to: {FIG_DIR}')
print(f'  - flow_time_series.png')
print(f'  - sensor_level_flow.png')
print(f'  - feature_distributions.png')
print(f'  - sensor_distributions.png')
print(f'  - temporal_patterns.png')
print(f'  - weekday_weekend_profile.png')
print(f'  - sensor_correlations.png')
print(f'  - adjacency_analysis.png')
print(f'  - flow_speed_diagram.png')
print(f'  - data_split.png')

PeMS04 EDA Summary
Dataset: PeMS04 (Caltrans District 4, SF Bay Area)
Period: January 1 - February 28, 2018 (59 days)
Shape: (16992, 307, 3) (timesteps, sensors, features)
Sampling: 5-minute intervals (12/hour, 288/day)

Features:
  Flow: [0, 919] veh/5min, mean=211.7
  Occupancy: [0.0000, 0.7716], mean=0.0528
  Speed: [3.0, 85.2] mph, mean=63.5

Data Quality:
  NaN values: 0
  Zero-flow observations: 82935 (1.59%)

Adjacency:
  Edges: 340 directed connections
  Distance range: [3.2, 2712.1]

Geographic Coordinates: NOT AVAILABLE in standard release

Forecasting Target: Traffic Flow (channel 0)

Figures saved to: C:\Users\Lenovo\OneDrive\Desktop\AUCML\traffic-congestion-ml\outputs\figures
  - flow_time_series.png
  - sensor_level_flow.png
  - feature_distributions.png
  - sensor_distributions.png
  - temporal_patterns.png
  - weekday_weekend_profile.png
  - sensor_correlations.png
  - adjacency_analysis.png
  - flow_speed_diagram.png
  - data_split.png
